# Acne Type Classifier
**EfficientNet-B3 / TensorFlow-Keras -- 3-class severity classification**

**Classes:** `acne-comedonica` · `acne-conglobata` · `acne-papulopustulosa`

**Dataset:** Roboflow `Acne type classification v3` -- 726 train / 28 test images

---
1. Install & Imports
2. Configuration
3. Dataset Exploration
4. Dataset & tf.data Pipelines
5. Model Definition
6. Training
7. Evaluation
8. Grad-CAM Visualization
9. Inference on a Single Image

## 1. Install & Imports

In [ ]:
# Run once -- skip if already installed
%pip install tensorflow scikit-learn matplotlib seaborn pillow tqdm -q

In [ ]:
import os
import random
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm.notebook import tqdm

import tensorflow as tf
from tensorflow.keras import layers

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_curve, auc, roc_auc_score,
)
from sklearn.preprocessing import label_binarize

warnings.filterwarnings('ignore')
print('TensorFlow:', tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print('GPUs available:', len(gpus))
for g in gpus:
    print(' ', g.name)

## 2. Configuration

In [ ]:
# -- Paths ---------------------------------------------------------------
DATA_ROOT    = 'acne dataset'
OUTPUT_DIR   = 'outputs'
VAL_FRACTION = 0.15

# -- Model ---------------------------------------------------------------
DROPOUT      = 0.4

# -- Training ------------------------------------------------------------
IMAGE_SIZE          = 224
BATCH_SIZE          = 32
EPOCHS              = 40
LR                  = 1e-3
WEIGHT_DECAY        = 1e-4
FREEZE_EPOCHS       = 5
WARMUP_EPOCHS       = 5
EARLY_STOP_PATIENCE = 10
IMBALANCE           = 'weighted'   # 'weighted' | 'none'
MIXUP_ALPHA         = 0.4          # set 0 to disable

SEED = 42
# -----------------------------------------------------------------------

os.makedirs(OUTPUT_DIR, exist_ok=True)

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

set_seed(SEED)

# Mixed precision (GPU only)
if gpus:
    tf.keras.mixed_precision.set_global_policy('mixed_float16')
    print('Mixed precision: float16')
else:
    print('No GPU found -- running on CPU')
print('Policy:', tf.keras.mixed_precision.global_policy())

## 3. Dataset Exploration

In [ ]:
IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

def list_images(folder):
    folder = Path(folder)
    return [p for p in folder.iterdir() if p.suffix.lower() in IMG_EXTS]

def get_class_names(root, split='train'):
    folder = Path(root) / split
    return sorted([d.name for d in folder.iterdir()
                   if d.is_dir() and not d.name.startswith('.')])

CLASS_NAMES = get_class_names(DATA_ROOT, 'train')
NUM_CLASSES = len(CLASS_NAMES)
print(f'Classes ({NUM_CLASSES}):', CLASS_NAMES)

for split in ('train', 'test'):
    counts = {cls: len(list_images(Path(DATA_ROOT) / split / cls))
              for cls in CLASS_NAMES}
    print(f'{split:5s}: {counts}  -> total {sum(counts.values())}')

train_counts = {cls: len(list_images(Path(DATA_ROOT) / 'train' / cls))
                for cls in CLASS_NAMES}
fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(CLASS_NAMES, train_counts.values(),
              color=sns.color_palette('muted', NUM_CLASSES))
ax.bar_label(bars)
ax.set_title('Training images per class')
ax.set_xlabel('Class'); ax.set_ylabel('Count')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'class_distribution.png'), dpi=150)
plt.show()

In [ ]:
N_PER_CLASS = 5
fig, axes = plt.subplots(NUM_CLASSES, N_PER_CLASS,
                          figsize=(3 * N_PER_CLASS, 3 * NUM_CLASSES))
for row, cls in enumerate(CLASS_NAMES):
    imgs = list_images(Path(DATA_ROOT) / 'train' / cls)[:N_PER_CLASS]
    for col in range(N_PER_CLASS):
        ax = axes[row][col]
        if col < len(imgs):
            ax.imshow(Image.open(imgs[col]).convert('RGB'))
        ax.axis('off')
        if col == 0:
            ax.set_title(cls.replace('acne-', ''), fontsize=10, fontweight='bold')
plt.suptitle('Sample images per class (train)', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'sample_grid.png'), dpi=150)
plt.show()

## 4. Dataset & tf.data Pipelines

In [ ]:
# -- Build sample lists ------------------------------------------------
def build_samples(root, split, class_names):
    samples = []
    for label, cls in enumerate(class_names):
        folder = Path(root) / split / cls
        for p in folder.iterdir():
            if p.suffix.lower() in IMG_EXTS:
                samples.append((str(p), label))
    return samples

def split_train_val(samples, val_fraction, seed=42):
    """Stratified split -- keeps class balance in val."""
    rng = random.Random(seed)
    by_class = {}
    for path, label in samples:
        by_class.setdefault(label, []).append((path, label))
    train_s, val_s = [], []
    for label, items in by_class.items():
        rng.shuffle(items)
        n_val = max(1, int(len(items) * val_fraction))
        val_s   += items[:n_val]
        train_s += items[n_val:]
    return train_s, val_s

all_train    = build_samples(DATA_ROOT, 'train', CLASS_NAMES)
test_samples = build_samples(DATA_ROOT, 'test',  CLASS_NAMES)
train_samples, val_samples = split_train_val(all_train, VAL_FRACTION, SEED)

print(f'Train: {len(train_samples)} | Val: {len(val_samples)} | Test: {len(test_samples)}')
for name, slist in [('Train', train_samples), ('Val', val_samples), ('Test', test_samples)]:
    counts = Counter(CLASS_NAMES[l] for _, l in slist)
    print(f'  {name}: {dict(counts)}')


# -- Class / sample weights for imbalance ------------------------------
train_labels     = [s[1] for s in train_samples]
label_counts     = Counter(train_labels)
total            = len(train_labels)
class_weight_arr = np.array(
    [total / (NUM_CLASSES * label_counts[i]) for i in range(NUM_CLASSES)],
    dtype=np.float32,
)
print('\nClass weights:', {CLASS_NAMES[i]: round(float(class_weight_arr[i]), 3)
                            for i in range(NUM_CLASSES)})


# -- Augmentation (applied on raw [0,255] pixels before normalisation) -
augment_layer = tf.keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomFlip('vertical'),
    layers.RandomRotation(15 / 360, fill_mode='reflect'),
    layers.RandomZoom((-0.1, 0.1), fill_mode='reflect'),
    layers.RandomTranslation(0.1, 0.1, fill_mode='reflect'),
    layers.RandomBrightness(0.3),
    layers.RandomContrast(0.3),
], name='augmentation')


# -- MixUp (batch-level, one-hot labels) --------------------------------
def mixup_batch(imgs, labels, weights):
    if MIXUP_ALPHA <= 0:
        return imgs, labels, weights
    lam = tf.random.uniform([], 0.0, 1.0)
    idx = tf.random.shuffle(tf.range(tf.shape(imgs)[0]))
    return (lam * imgs    + (1 - lam) * tf.gather(imgs,    idx),
            lam * labels  + (1 - lam) * tf.gather(labels,  idx),
            lam * weights + (1 - lam) * tf.gather(weights, idx))


# -- tf.data pipeline --------------------------------------------------
_PREPROCESS = tf.keras.applications.efficientnet.preprocess_input

def _read_raw(path):
    raw = tf.io.read_file(path)
    img = tf.io.decode_image(raw, channels=3, expand_animations=False)
    img.set_shape([None, None, 3])
    return tf.cast(img, tf.float32)

def build_dataset(samples, sample_weights, training=False):
    paths   = [s[0] for s in samples]
    labels  = tf.keras.utils.to_categorical([s[1] for s in samples], NUM_CLASSES)
    weights = np.array(sample_weights, dtype=np.float32)

    ds = tf.data.Dataset.from_tensor_slices((paths, labels, weights))

    if training:
        def _load(path, label, w):
            img = _read_raw(path)
            img = tf.image.resize(img, [IMAGE_SIZE + 20, IMAGE_SIZE + 20])
            img = tf.image.random_crop(img, [IMAGE_SIZE, IMAGE_SIZE, 3])
            return img, label, w

        ds = (ds
              .shuffle(len(samples), seed=SEED, reshuffle_each_iteration=True)
              .map(_load, num_parallel_calls=tf.data.AUTOTUNE)
              .batch(BATCH_SIZE, drop_remainder=False))

        def _aug_norm(imgs, labels, weights):
            imgs = augment_layer(imgs, training=True)
            imgs = tf.clip_by_value(imgs, 0.0, 255.0)
            imgs = _PREPROCESS(imgs)
            return imgs, labels, weights

        ds = (ds
              .map(_aug_norm,    num_parallel_calls=tf.data.AUTOTUNE)
              .map(mixup_batch,  num_parallel_calls=tf.data.AUTOTUNE))
    else:
        def _load_val(path, label, w):
            img = _read_raw(path)
            img = tf.image.resize(img, [IMAGE_SIZE, IMAGE_SIZE])
            img = _PREPROCESS(img)
            return img, label, w

        ds = (ds
              .map(_load_val, num_parallel_calls=tf.data.AUTOTUNE)
              .batch(BATCH_SIZE, drop_remainder=False))

    return ds.prefetch(tf.data.AUTOTUNE)


if IMBALANCE == 'weighted':
    train_sw = [float(class_weight_arr[l]) for _, l in train_samples]
else:
    train_sw = [1.0] * len(train_samples)

train_ds = build_dataset(train_samples, train_sw,                  training=True)
val_ds   = build_dataset(val_samples,   [1.0] * len(val_samples),  training=False)
test_ds  = build_dataset(test_samples,  [1.0] * len(test_samples), training=False)

print(f'\nBatches -> train: {len(train_ds)} | val: {len(val_ds)} | test: {len(test_ds)}')

## 5. Model Definition

In [ ]:
def build_model(num_classes, dropout=0.4):
    """
    Flat functional model (base layers not nested) so every layer
    is reachable by name for GradCAM.
    """
    base = tf.keras.applications.EfficientNetB3(
        include_top=False,
        weights='imagenet',
        input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3),
    )
    x   = base.output                                            # feature map
    x   = layers.GlobalAveragePooling2D(name='gap')(x)
    x   = layers.Dropout(dropout, name='head_dropout')(x)
    # float32 output required for mixed-precision stability
    out = layers.Dense(num_classes, name='predictions',
                       dtype='float32')(x)
    model = tf.keras.Model(inputs=base.input, outputs=out)
    return model, base


model, base_model = build_model(NUM_CLASSES, DROPOUT)
print(f'Backbone   : EfficientNetB3')
print(f'Parameters : {model.count_params():,} total')
print(f'Output     : {model.output_shape}')

# Verify GradCAM target layer exists
LAST_CONV_LAYER = 'top_activation'
try:
    lyr = model.get_layer(LAST_CONV_LAYER)
    print(f'GradCAM layer "{LAST_CONV_LAYER}": {lyr.output_shape}')
except ValueError:
    print(f'Layer "{LAST_CONV_LAYER}" not found -- listing last 5 layers:')
    for l in model.layers[-5:]:
        print(f'  {l.name}: {l.output_shape}')

## 6. Training

In [ ]:
# -- LR schedule: linear warmup then cosine annealing -----------------
class WarmupCosineSchedule(tf.keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, base_lr, warmup_steps, total_steps):
        self.base_lr      = float(base_lr)
        self.warmup_steps = float(warmup_steps)
        self.total_steps  = float(total_steps)

    def __call__(self, step):
        step      = tf.cast(step, tf.float32)
        warmup_lr = self.base_lr * (step + 1.0) / self.warmup_steps
        decay     = self.total_steps - self.warmup_steps
        cos_arg   = np.pi * tf.minimum(step - self.warmup_steps, decay) / (decay + 1e-8)
        cosine_lr = self.base_lr * 0.5 * (1.0 + tf.cos(cos_arg))
        return tf.where(step < self.warmup_steps, warmup_lr, cosine_lr)

    def get_config(self):
        return {'base_lr': self.base_lr,
                'warmup_steps': self.warmup_steps,
                'total_steps': self.total_steps}


# -- Callbacks ---------------------------------------------------------
ckpt_loss = tf.keras.callbacks.ModelCheckpoint(
    os.path.join(OUTPUT_DIR, 'best_loss.weights.h5'),
    monitor='val_loss', save_best_only=True, mode='min',
    save_weights_only=True, verbose=1,
)
ckpt_acc = tf.keras.callbacks.ModelCheckpoint(
    os.path.join(OUTPUT_DIR, 'best_acc.weights.h5'),
    monitor='val_accuracy', save_best_only=True, mode='max',
    save_weights_only=True, verbose=0,
)
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=EARLY_STOP_PATIENCE,
    restore_best_weights=True, verbose=1,
)
csv_log = tf.keras.callbacks.CSVLogger(
    os.path.join(OUTPUT_DIR, 'training_log.csv'), append=True,
)

print('Callbacks ready.')

In [ ]:
steps = len(train_ds)   # batches per epoch

# ---- Phase 1: Head only (backbone frozen) ----------------------------
base_model.trainable = False
n_trainable = sum(np.prod(v.shape) for v in model.trainable_variables)
print(f'Phase 1: backbone frozen -- {n_trainable:,} trainable params (head only)\n')

model.compile(
    optimizer=tf.keras.optimizers.AdamW(LR, weight_decay=WEIGHT_DECAY),
    loss=tf.keras.losses.CategoricalCrossentropy(
        from_logits=True, label_smoothing=0.1),
    metrics=[tf.keras.metrics.CategoricalAccuracy(name='accuracy')],
)

hist1 = model.fit(
    train_ds, validation_data=val_ds,
    epochs=FREEZE_EPOCHS,
    callbacks=[ckpt_loss, ckpt_acc, csv_log],
    verbose=1,
)

# ---- Phase 2: Fine-tune full model -----------------------------------
base_model.trainable = True
n_trainable = sum(np.prod(v.shape) for v in model.trainable_variables)
print(f'\nPhase 2: backbone unfrozen -- {n_trainable:,} trainable params')

# Backbone gets LR/10 via warmup+cosine; head retains same schedule
warmup_steps = WARMUP_EPOCHS * steps
total_steps  = (EPOCHS - FREEZE_EPOCHS) * steps
lr_schedule  = WarmupCosineSchedule(LR / 10, warmup_steps, total_steps)

model.compile(
    optimizer=tf.keras.optimizers.AdamW(lr_schedule, weight_decay=WEIGHT_DECAY),
    loss=tf.keras.losses.CategoricalCrossentropy(
        from_logits=True, label_smoothing=0.1),
    metrics=[tf.keras.metrics.CategoricalAccuracy(name='accuracy')],
)

hist2 = model.fit(
    train_ds, validation_data=val_ds,
    initial_epoch=FREEZE_EPOCHS,
    epochs=EPOCHS,
    callbacks=[ckpt_loss, ckpt_acc, early_stop, csv_log],
    verbose=1,
)

# Combine histories for plotting
history = {
    'train_loss': hist1.history['loss']         + hist2.history['loss'],
    'val_loss':   hist1.history['val_loss']      + hist2.history['val_loss'],
    'train_acc':  hist1.history['accuracy']      + hist2.history['accuracy'],
    'val_acc':    hist1.history['val_accuracy']  + hist2.history['val_accuracy'],
}
print('\nTraining complete.')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(history['train_loss'], label='Train')
axes[0].plot(history['val_loss'],   label='Val')
axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend()

axes[1].plot(history['train_acc'], label='Train')
axes[1].plot(history['val_acc'],   label='Val')
axes[1].set_title('Accuracy'); axes[1].set_xlabel('Epoch'); axes[1].legend()

plt.suptitle('EfficientNetB3 (TF/Keras) -- Training History', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'training_curves.png'), dpi=150)
plt.show()

## 7. Evaluation

In [ ]:
# Load best-loss checkpoint
model.load_weights(os.path.join(OUTPUT_DIR, 'best_loss.weights.h5'))
print('Loaded best_loss.weights.h5')

# Predict on test set
test_logits = model.predict(test_ds, verbose=0)
test_probs  = tf.nn.softmax(test_logits).numpy()
test_preds  = test_probs.argmax(axis=1)

# True integer labels (from sample list -- avoids iterating the dataset twice)
test_labels_np = np.array([s[1] for s in test_samples])

test_acc = (test_preds == test_labels_np).mean()
print(f'\nTest Accuracy: {test_acc:.4f} ({test_acc * 100:.1f}%)')
print('\nClassification Report:')
print(classification_report(test_labels_np, test_preds, target_names=CLASS_NAMES))

In [ ]:
SHORT = [c.replace('acne-', '') for c in CLASS_NAMES]
cm    = confusion_matrix(test_labels_np, test_preds)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ConfusionMatrixDisplay(cm, display_labels=SHORT).plot(
    ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Confusion Matrix (counts)')

cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=SHORT, yticklabels=SHORT, ax=axes[1])
axes[1].set_title('Confusion Matrix (normalised)')
axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('True')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'confusion_matrix.png'), dpi=150)
plt.show()

### ROC Curves & AUC Scores

In [ ]:
test_labels_bin = label_binarize(test_labels_np, classes=list(range(NUM_CLASSES)))

fig, ax = plt.subplots(figsize=(7, 5))
colors  = sns.color_palette('muted', NUM_CLASSES)

for i, cls in enumerate(CLASS_NAMES):
    fpr, tpr, _ = roc_curve(test_labels_bin[:, i], test_probs[:, i])
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=colors[i], lw=2,
            label=f"{cls.replace('acne-', '')} (AUC = {roc_auc:.2f})")

ax.plot([0, 1], [0, 1], 'k--', lw=1)
ax.set_xlim([0, 1]); ax.set_ylim([0, 1.02])
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves (test set)'); ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'roc_curves.png'), dpi=150)
plt.show()

macro_auc = roc_auc_score(test_labels_bin, test_probs, average='macro')
print(f'Macro AUC: {macro_auc:.4f}')

### Confidence Calibration

In [ ]:
def reliability_diagram(probs, labels, n_bins=10, ax=None, title='Reliability Diagram'):
    confidences = probs.max(axis=1)
    predictions = probs.argmax(axis=1)
    correct     = (predictions == labels).astype(float)
    bin_edges   = np.linspace(0, 1, n_bins + 1)
    bin_acc, bin_conf, bin_counts = [], [], []

    for lo, hi in zip(bin_edges[:-1], bin_edges[1:]):
        mask = (confidences >= lo) & (confidences < hi)
        if mask.sum() > 0:
            bin_acc.append(correct[mask].mean())
            bin_conf.append(confidences[mask].mean())
            bin_counts.append(int(mask.sum()))
        else:
            bin_acc.append(None)
            bin_conf.append((lo + hi) / 2)
            bin_counts.append(0)

    if ax is None:
        _, ax = plt.subplots(figsize=(5, 5))
    valid = [(c, a, n) for c, a, n in zip(bin_conf, bin_acc, bin_counts) if a is not None]
    if valid:
        x, y, _ = zip(*valid)
        ax.bar(x, y, width=0.1, alpha=0.7, color='steelblue', label='Model')
    ax.plot([0, 1], [0, 1], 'k--', label='Perfect calibration')
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.set_xlabel('Confidence'); ax.set_ylabel('Accuracy')
    ax.set_title(title); ax.legend()

    ece = sum((n / len(labels)) * abs(a - c)
              for c, a, n in zip(bin_conf, bin_acc, bin_counts) if a is not None)
    ax.text(0.05, 0.92, f'ECE = {ece:.4f}', transform=ax.transAxes, fontsize=10)
    return ece


fig, ax = plt.subplots(figsize=(5, 5))
ece = reliability_diagram(test_probs, test_labels_np, ax=ax,
                          title='Reliability Diagram (EfficientNetB3 TF/Keras)')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'calibration.png'), dpi=150)
plt.show()
print(f'ECE: {ece:.4f}  (lower is better, 0 = perfect)')

## 8. Grad-CAM Visualization

In [ ]:
# -- Custom Grad-CAM via GradientTape ----------------------------------
def get_gradcam_heatmap(model, img_array, layer_name=LAST_CONV_LAYER, pred_index=None):
    """img_array: (1, H, W, 3) preprocessed float32 tensor."""
    grad_model = tf.keras.Model(
        inputs  = model.inputs,
        outputs = [model.get_layer(layer_name).output, model.output],
    )
    with tf.GradientTape() as tape:
        conv_out, preds = grad_model(img_array, training=False)
        if pred_index is None:
            pred_index = int(tf.argmax(preds[0]))
        score = preds[:, pred_index]
    grads   = tape.gradient(score, conv_out)
    pooled  = tf.reduce_mean(grads, axis=(0, 1, 2))
    heatmap = tf.reduce_sum(tf.nn.relu(conv_out[0] * pooled), axis=-1)
    heatmap = heatmap / (tf.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy(), pred_index


def overlay_heatmap(raw_img_01, heatmap, alpha=0.5):
    """raw_img_01: float32 [0,1] RGB array. Returns blended overlay."""
    h, w = raw_img_01.shape[:2]
    hm   = np.array(Image.fromarray((heatmap * 255).astype(np.uint8))
                    .resize((w, h), Image.BILINEAR)) / 255.0
    cmap = plt.cm.jet(hm)[..., :3]
    return np.clip((1 - alpha) * raw_img_01 + alpha * cmap, 0, 1)


def preprocess_single(path):
    """Returns (preprocessed_batch, raw_PIL_image)."""
    img = Image.open(path).convert('RGB').resize((IMAGE_SIZE, IMAGE_SIZE))
    arr = np.array(img, dtype=np.float32)
    inp = tf.keras.applications.efficientnet.preprocess_input(arr)[np.newaxis]
    return inp, img


# -- Collect one correct + one wrong per class -------------------------
correct_by_class = {i: None for i in range(NUM_CLASSES)}
wrong_by_class   = {i: None for i in range(NUM_CLASSES)}

for path, true_lbl in test_samples:
    inp, raw_img = preprocess_single(path)
    heatmap, pred = get_gradcam_heatmap(model, inp)
    entry = (raw_img, heatmap, true_lbl, pred)
    if pred == true_lbl and correct_by_class[true_lbl] is None:
        correct_by_class[true_lbl] = entry
    elif pred != true_lbl and wrong_by_class[true_lbl] is None:
        wrong_by_class[true_lbl]   = entry
    if (all(v is not None for v in correct_by_class.values()) and
            all(v is not None for v in wrong_by_class.values())):
        break

cols = NUM_CLASSES * 2
fig, axes = plt.subplots(2, cols, figsize=(4 * cols, 8))

for cls_i in range(NUM_CLASSES):
    for row, bucket in enumerate([correct_by_class, wrong_by_class]):
        col   = cls_i * 2 + row
        ax    = axes[row][col]
        entry = bucket[cls_i]
        if entry is None:
            ax.axis('off'); continue
        raw_img, heatmap, true_lbl, pred = entry
        raw_01  = np.array(raw_img, dtype=np.float32) / 255.0
        overlay = overlay_heatmap(raw_01, heatmap)
        ax.imshow(overlay)
        colour = 'green' if pred == true_lbl else 'red'
        label  = 'Correct' if pred == true_lbl else 'Wrong'
        ax.set_title(
            f"{label}\nTrue: {CLASS_NAMES[true_lbl].replace('acne-', '')}\n"
            f"Pred: {CLASS_NAMES[pred].replace('acne-', '')}",
            color=colour, fontsize=8,
        )
        ax.axis('off')

plt.suptitle('Grad-CAM per class: correct (left) vs wrong (right) | green=correct  red=wrong',
             fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'gradcam_per_class.png'), dpi=150)
plt.show()

## 9. Inference on a Single Image

In [ ]:
def predict(image_path: str) -> dict:
    """Predict acne type for a single image."""
    inp, raw_img = preprocess_single(image_path)
    logits = model.predict(inp, verbose=0)
    probs  = tf.nn.softmax(logits).numpy().squeeze()

    pred_idx   = int(probs.argmax())
    confidence = float(probs[pred_idx])
    result = {
        'predicted_class': CLASS_NAMES[pred_idx],
        'confidence':      confidence,
        'probabilities':   {c: float(p) for c, p in zip(CLASS_NAMES, probs)},
    }

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
    ax1.imshow(raw_img)
    ax1.axis('off')
    ax1.set_title(
        f"Prediction: {result['predicted_class'].replace('acne-', '')}\n"
        f"Confidence: {confidence:.1%}", fontsize=12,
    )
    short_names = [c.replace('acne-', '') for c in CLASS_NAMES]
    colours = ['#4CAF50' if i == pred_idx else '#90CAF9' for i in range(NUM_CLASSES)]
    ax2.barh(short_names, probs, color=colours)
    ax2.set_xlim(0, 1); ax2.set_xlabel('Probability')
    ax2.set_title('Class Probabilities')
    for i, p in enumerate(probs):
        ax2.text(p + 0.01, i, f'{p:.1%}', va='center', fontsize=9)
    plt.tight_layout()
    plt.show()
    return result


# Demo: random test image
sample_path, sample_label = random.choice(test_samples)
print(f'Ground truth: {CLASS_NAMES[sample_label]}')
result = predict(sample_path)
print(result)